In [1]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [8]:
SEED = 42
rng = np.random.default_rng(SEED)

START_DATE = pd.Timestamp("2026-01-01")
END_DATE = pd.Timestamp("2026-06-30")

NUM_RIDERS = 25_000
NUM_AGENTS = 120
NUM_TRIPS = 120_000

In [9]:
# Create the output directory for the tables if it does not exist

PROJECT_ROOT = Path("__file__").resolve().parents[1]
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
# Table: riders

def generate_riders():
    cities = ["Metroville", "Lakewood", "Riverton", "Hillview"]
    customer_segments = ["Occasional", "Regular", "Frequent"]

    rider_ids = [f"R{str(i).zfill(6)}" for i in range(1, NUM_RIDERS + 1)]
    
    signup_start = pd.Timestamp("2024-01-01")
    signup_days = (START_DATE - signup_start).days
    signup_dates = (
        signup_start
        + pd.to_timedelta(
            rng.integers(
                0,
                signup_days + 1,
                size=NUM_RIDERS,
            ),
            unit="D"
        )
    )

    riders = pd.DataFrame({
        "rider_id": rider_ids,
        "signup_date": signup_dates,
        "home_city": rng.choice(cities, size=NUM_RIDERS, p=[0.35, 0.25, 0.22, 0.18]),
        "customer_segment": rng.choice(customer_segments, size=NUM_RIDERS, p=[0.45, 0.35, 0.20])
    })

    return riders

In [21]:
# Function to save generated tables

def save_dataframe(df, filename):
    output_path = RAW_DATA_DIR / filename
    df.to_csv(output_path, index=False)

    print(f"Saved {filename}: {len(df)} rows")

In [22]:
riders = generate_riders()
save_dataframe(riders, "riders.csv")

Saved riders.csv: 25000 rows


In [26]:
# Table: agents

def generate_agents():
    support_teams = ["General Support", "Payments", "AV Specialist", "Safety"]
    agent_ids = [f"A{str(i).zfill(4)}" for i in range(1, NUM_AGENTS + 1)]
    teams = rng.choice(support_teams, size=NUM_AGENTS, p=[0.50, 0.20, 0.20, 0.10])
    tenure_months = rng.integers(1, 73, size=NUM_AGENTS)

    cost_per_hour = []

    for team in teams:
        if team == "General Support":
            hourly_cost = rng.uniform(18, 24)
        elif team == "Payments":
            hourly_cost = rng.uniform(22, 28)
        elif team == "AV Specialist":
            hourly_cost = rng.uniform(25, 34)
        else:
            hourly_cost = rng.uniform(28, 40)

        cost_per_hour.append(round(hourly_cost, 2))

    agents = pd.DataFrame({
        "agent_id": agent_ids,
        "support_team": teams,
        "tenure_months": tenure_months,
        "cost_per_hour": cost_per_hour,
        "location": rng.choice(["Accra Hub", "Remote"], size=NUM_AGENTS, p=[0.60, 0.40]),
        "active_flag": rng.choice([True, False], size=NUM_AGENTS, p=[0.95, 0.05])
    })

    return agents

In [27]:
agents = generate_agents()
save_dataframe(agents, "agents.csv")

Saved agents.csv: 120 rows


In [33]:
agents["support_team"].value_counts()

support_team
General Support    57
Payments           30
AV Specialist      21
Safety             12
Name: count, dtype: int64